# FINE-TUNNING FOR OPENVLA

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# FINE-TUNNING SCRIPT FOR OPENVLA MODEL PART 1
# CONFIGURATIONS

import os
os.chdir("/home/ids/ext-5219/tokenizer/openvla-oft/")  # replace with your repo root
print("Current working directory:", os.getcwd())

sys.argv.append("pusht")

from collections import deque
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import draccus
import torch
import torch.distributed as dist
import tqdm
from accelerate import PartialState
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig
from transformers import AutoConfig, AutoImageProcessor
from transformers.modeling_outputs import CausalLMOutputWithPast

import wandb
from prismatic.models.backbones.llm.prompting import PurePromptBuilder, VicunaV15ChatPromptBuilder
from prismatic.util.data_utils import PaddedCollatorForActionPrediction
from prismatic.vla.subtrajectory_tokenizer import SubtrajectoryTokenizer
from prismatic.vla.datasets.datasets_custom import RLDSCustomBatchTransform, RLDSDatasetCustom
from prismatic.vla.datasets.rlds.utils.data_utils import save_dataset_statistics

from prismatic.extern.hf.configuration_prismatic import OpenVLAConfig
from prismatic.extern.hf.modeling_prismatic import OpenVLAForActionPrediction
from prismatic.extern.hf.processing_prismatic import PrismaticImageProcessor, PrismaticProcessor

# Sane Defaults
os.environ["TOKENIZERS_PARALLELISM"] = "false"


# # === Utilities ===
# # fmt: off
# def create_vision_transform(vla: nn.Module, input_size: int) -> Callable[[Image.Image], torch.Tensor]:
#     """Gets image transform for the vision encoder."""
#     data_cfg = timm.data.resolve_model_data_config(vla.vision_backbone)
#     data_cfg["input_size"] = (3, input_size, input_size)
#     return timm.data.create_transform(
#         input_size=data_cfg["input_size"],
#         interpolation=data_cfg["interpolation"],
#         mean=data_cfg["mean"],
#         std=data_cfg["std"],
#         crop_pct=1.0,           # Set to 1.0 to disable cropping
#         crop_mode="center",     # Default crop mode --> no-op when `crop_pct == 1.0`
#         is_training=False,      # Disable image_aug when loading transform; handled by RLDS dataloader
#     )
#
# # fmt: on



# fmt: off
vla_path: str = "openvla/openvla-7b"                            # Path to OpenVLA model (on HuggingFace Hub)

# Directory Paths
data_root_dir: Path = Path("/home/ids/ext-5219/tokenizer/test")        # Path to Open-X dataset directory
dataset_name: str = "columbia_cairlab_pusht_real"                                # Name of fine-tuning dataset (e.g., `droid_wipe`)
run_root_dir: Path = Path("runs")                               # Path to directory to store logs & checkpoints
adapter_tmp_dir: Path = Path("adapter-tmp")                     # Temporary directory for LoRA weights before fusing

# Fine-tuning Parameters
batch_size: int = 8                                            # Fine-tuning batch size
max_steps: int = 1000 #200_000                                       # Max number of fine-tuning steps
save_steps: int = 500  #5000                                        # Interval for checkpoint saving
learning_rate: float = 2e-5#5e-4                                     # Fine-tuning learning rate
grad_accumulation_steps: int = 1                                # Gradient accumulation steps
image_aug: bool = False                                          # Whether to train with image augmentations
shuffle_buffer_size: int = 10000    #100_000                          # Dataloader shuffle buffer size (can reduce if OOM)
save_latest_checkpoint_only: bool = True                        # Whether to save only one checkpoint per run and
                                                                #   continually overwrite the latest checkpoint
                                                                #   (If False, saves all checkpoints)

# LoRA Arguments
use_lora: bool = True                                           # Whether to use LoRA fine-tuning
lora_rank: int = 32                                             # Rank of LoRA weight matrix
lora_dropout: float = 0.0                                       # Dropout applied to LoRA weights
use_quantization: bool = False                                  # Whether to 4-bit quantize VLA for LoRA fine-tuning
                                                                #   => CAUTION: Reduces memory but hurts performance

# Tracking Parameters
wandb_entity: str = "pollen"          # Name of WandB entity
wandb_project: str = "openvla"        # Name of WandB project                         # Name of entity to log under
run_id_note: Optional[str] = None                               # Extra note for logging, Weights & Biases

# fmt: on


Current working directory: /home/ids/ext-5219/tokenizer/openvla-oft


/home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-03 14:38:42.187853: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-03 14:38:42.221354: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-03 14:38:42.221406: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026

Using PUSHT constants:
  NUM_ACTIONS_CHUNK = 1
  ACTION_DIM = 1
  PROPRIO_DIM = 8
  ACTION_PROPRIO_NORMALIZATION_TYPE = bounds_q99
If needed, manually set the correct constants in `prismatic/vla/constants.py`!


2026-02-03 14:38:47.692320: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2348] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 9.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


In [3]:
# FINE-TUNNING SCRIPT FOR OPENVLA MODEL PART 2
# PARAMETERS FOR MODEL 


print(f"Fine-tuning OpenVLA Model `{vla_path}` on `{dataset_name}`")

# [Validate] Ensure GPU Available & Set Device / Distributed Context
assert torch.cuda.is_available(), "Fine-tuning assumes at least one GPU is available!"
distributed_state = PartialState()
# torch.cuda.set_device(device_id := distributed_state.local_process_index)
device_id=0 #I will use only one GPU for the momment
device_id = distributed_state.local_process_index
torch.cuda.set_device(device_id)
torch.cuda.empty_cache()

# Configure Unique Experiment ID & Log Directory
exp_id = (
    f"{vla_path.split('/')[-1]}+{dataset_name}"
    f"+b{batch_size * grad_accumulation_steps}"
    f"+lr-{learning_rate}"
    f"+test"
)
if use_lora:
    exp_id += f"+lora-r{lora_rank}+dropout-{lora_dropout}"
if use_quantization:
    exp_id += "+q-4bit"
if run_id_note is not None:
    exp_id += f"--{run_id_note}"
if image_aug:
    exp_id += "--image_aug"

# Start =>> Build Directories
run_dir, adapter_dir = run_root_dir / exp_id, adapter_tmp_dir / exp_id
os.makedirs(run_dir, exist_ok=True)

# Quantization Config =>> only if LoRA fine-tuning
quantization_config = None
if use_quantization:
    assert use_lora, "Quantized training only supported for LoRA fine-tuning!"
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4"
    )

# Register OpenVLA model to HF Auto Classes (not needed if the model is on HF Hub)
AutoConfig.register("openvla", OpenVLAConfig)
AutoImageProcessor.register(OpenVLAConfig, PrismaticImageProcessor)
AutoProcessor.register(OpenVLAConfig, PrismaticProcessor)
AutoModelForVision2Seq.register(OpenVLAConfig, OpenVLAForActionPrediction)

# Load OpenVLA Processor and Model using HF AutoClasses
processor = AutoProcessor.from_pretrained(vla_path, trust_remote_code=True)
vla = AutoModelForVision2Seq.from_pretrained(
    vla_path,
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

# Device Placement =>> note that BitsAndBytes automatically handles for quantized training
if use_quantization:
    vla = prepare_model_for_kbit_training(vla)
else:
    vla = vla.to(device_id)

    # [LoRA] Wrap Model w/ PEFT `LoraConfig` =>> by default we set `target_modules=all-linear`
    if use_lora:
        lora_config = LoraConfig(
            r=lora_rank,
            lora_alpha=min(lora_rank, 16),
            lora_dropout=lora_dropout,
            target_modules="all-linear",
            init_lora_weights="gaussian",
        )
        vla = get_peft_model(vla, lora_config)
        vla.print_trainable_parameters()

    # Wrap VLA in PyTorch DDP Wrapper for Multi-GPU Training
    # vla = DDP(vla, device_ids=[device_id], find_unused_parameters=True, gradient_as_bucket_view=True)

    # Create Optimizer =>> note that we default to a simple constant learning rate!
    trainable_params = [param for param in vla.parameters() if param.requires_grad]
    optimizer = AdamW(trainable_params, lr=learning_rate)

    # Create Action Tokenizer
    action_tokenizer = SubtrajectoryTokenizer(processor.tokenizer,bins=30,min_action=0,max_action=30)

Fine-tuning OpenVLA Model `openvla/openvla-7b` on `columbia_cairlab_pusht_real`


Loading checkpoint shards: 100%|██████████████████████████| 3/3 [00:00<00:00,  4.53it/s]


trainable params: 110,828,288 || all params: 7,652,065,472 || trainable%: 1.4483


In [4]:
# FINE-TUNNING SCRIPT FOR OPENVLA MODEL PART 3
# LOADING DATASET
# Create training and optional validation datasets

# Load Fine-tuning Dataset =>> note that we use an RLDS-formatted dataset following Open X-Embodiment by default.
#   =>> If you want to use a non-RLDS dataset (e.g., a standard PyTorch Dataset) see the following commented block.
#   =>> Note that our training code does not loop over epochs because the RLDS loader does this implicitly; if using
#       your own Dataset, make sure to add the appropriate logic to the training loop!
#
# ---
# from prismatic.vla.datasets import DummyDataset
#
# vla_dataset = DummyDataset(
#     action_tokenizer,
#     processor.tokenizer,
#     image_transform=processor.image_processor.apply_transform,
#     prompt_builder_fn=PurePromptBuilder if "v01" not in vla_path else VicunaV15ChatPromptBuilder,
# )
# ---
batch_transform = RLDSCustomBatchTransform(
    action_tokenizer,
    processor.tokenizer,
    image_transform=processor.image_processor.apply_transform,
    prompt_builder_fn=PurePromptBuilder if "v01" not in vla_path else VicunaV15ChatPromptBuilder,
)
vla_dataset = RLDSDatasetCustom(
    data_root_dir,
    dataset_name,
    batch_transform,
    resize_resolution=tuple(vla.config.image_sizes),
    shuffle_buffer_size=shuffle_buffer_size,
    image_aug=image_aug,
)

# [Important] Save Dataset Statistics =>> used to de-normalize actions for inference!
if distributed_state.is_main_process:
    save_dataset_statistics(vla_dataset.dataset_statistics, run_dir)

# Create Collator and DataLoader
collator = PaddedCollatorForActionPrediction(
    processor.tokenizer.model_max_length, processor.tokenizer.pad_token_id, padding_side="right"
)
dataloader = DataLoader(
    vla_dataset,
    batch_size=batch_size,
    sampler=None,
    collate_fn=collator,
    num_workers=0,  # Important =>> Set to 0 if using RLDS; TFDS rolls its own parallelism!
)

# Initialize Logging =>> W&B
if distributed_state.is_main_process:
    wandb.init(entity=wandb_entity, project=wandb_project, name=f"ft+{exp_id}")

2026-02-03 14:39:00.566570: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization


02/03 [14:39:00] INFO     | >> [*] Loading existing dataset statistics from                       ]8;id=472461;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/utils/data_utils.py\data_utils.py]8;;\:]8;id=638300;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/utils/data_utils.py#199\199]8;;\
                          /home/ids/ext-5219/tokenizer/test/columbia_cairlab_pusht_real/1.0.0/dat                  
                          aset_statistics_d6170bf2de88fd222da6c9a2203ee8e1f88e82227a970154e370e5e                  
                          137360b3e.json.                                                                          

2026-02-03 14:39:00.905416: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization



######################################################################################
# Loading the following 1 datasets (incl. sampling weight):                         #
# columbia_cairlab_pusht_real: =============================================1.000000 #
######################################################################################



02/03 [14:39:01] INFO     | >> [*] Threads per Dataset: [1]                                          ]8;id=623190;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py\dataset.py]8;;\:]8;id=856049;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py#538\538]8;;\

...

                 INFO     | >> [*] Constructing datasets...                                          ]8;id=859394;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py\dataset.py]8;;\:]8;id=331557;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py#542\542]8;;\

2026-02-03 14:39:01.267756: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization


02/03 [14:39:02] INFO     | >> [*] Applying frame transforms on dataset...                           ]8;id=460169;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py\dataset.py]8;;\:]8;id=105238;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py#582\582]8;;\

                 INFO     | >> [*] Saved dataset statistics file at path                          ]8;id=399646;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/utils/data_utils.py\data_utils.py]8;;\:]8;id=322711;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/utils/data_utils.py#284\284]8;;\
                          runs/openvla-7b+columbia_cairlab_pusht_real+b8+lr-2e-05+test+lora-r32+d                  
                          ropout-0.0/dataset_statistics.json                                                       

wandb: Currently logged in as: cataclysme-apocalypse (pollen) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
# FINE-TUNNING SCRIPT FOR OPENVLA OFT MODEL PART 4
#TRAINING LOOP  # ==================================================

# Deque to store recent train metrics (used for computing smoothened metrics for gradient accumulation)
recent_losses = deque(maxlen=grad_accumulation_steps)
recent_action_accuracies = deque(maxlen=grad_accumulation_steps)
recent_l1_losses = deque(maxlen=grad_accumulation_steps)

# Train!
with tqdm.tqdm(total=max_steps, leave=False) as progress:
    vla.train()
    optimizer.zero_grad()
    for batch_idx, batch in enumerate(dataloader):
        with torch.autocast("cuda", dtype=torch.bfloat16):
            output: CausalLMOutputWithPast = vla(
                input_ids=batch["input_ids"].to(device_id),
                attention_mask=batch["attention_mask"].to(device_id),
                pixel_values=batch["pixel_values"].to(torch.bfloat16).to(device_id),
                labels=batch["labels"],
            )
            loss = output.loss

        # Normalize loss to account for gradient accumulation
        normalized_loss = loss / grad_accumulation_steps

        # Backward pass
        normalized_loss.backward()

        # Compute Accuracy and L1 Loss for Logging
        action_logits = output.logits[:, vla.vision_backbone.featurizer.patch_embed.num_patches : -1]
   
        action_preds = action_logits.argmax(dim=2)
        action_gt = batch["labels"][:, 1:].to(action_preds.device)
 
        # print(f"--- DEBUG: ACCTION GT {action_gt} ---")
        mask = action_gt > action_tokenizer.action_token_begin_idx

        # print(f"--- DEBUG: pred shape {action_preds.shape} ---")
        # print(f"--- DEBUG: gt shape {action_gt.shape} ---")

        # print(f"--- DEBUG: mask  {mask} ---")
        # print(f"--- DEBUG: TOKEN DEBUT {action_tokenizer.action_token_begin_idx} ---")

        # print(f"--- DEBUG: GT TOKEN {action_gt} ---")

        # Compute Accuracy
        correct_preds = (action_preds == action_gt) & mask
        action_accuracy = correct_preds.sum().float() / mask.sum().float()
        # print(f"--- DEBUG: ACCURACY {action_accuracy} ---")


        # Compute L1 Loss on Predicted (Continuous) Actions
        subtraject_ID_pred = torch.tensor(
            action_tokenizer.decode_token_ids_to_actions(action_preds[mask].cpu().numpy())
        )
        subtraject_ID_pred_gt = torch.tensor(
            action_tokenizer.decode_token_ids_to_actions(action_gt[mask].cpu().numpy())
        )

        logits = action_logits.transpose(1, 2)
        logits_masked=action_logits[mask]
        action_gt_masked=action_gt[mask]
        # print(f"--- DEBUG: action_logits shape {action_preds.shape} ---")
        # print(f"--- DEBUG: action_logits shape {action_preds[mask]} ---") 
        # print(f"--- DEBUG: GT shape {action_gt_masked} ---")
        # action_crossEntropy_loss = torch.nn.functional.cross_entropy(logits, action_gt)
        action_crossEntropy_loss = torch.nn.functional.cross_entropy(logits_masked, action_gt_masked)

        # Store recent train metrics
        recent_losses.append(loss.item())
        recent_action_accuracies.append(action_accuracy.item())
        recent_l1_losses.append(action_crossEntropy_loss.item())

        # Compute gradient step index
        gradient_step_idx = batch_idx // grad_accumulation_steps

        # Compute smoothened train metrics
        #   =>> Equal to current step metrics when not using gradient accumulation
        #   =>> Otherwise, equal to the average of metrics observed over micro-batches used for gradient accumulation
        smoothened_loss = sum(recent_losses) / len(recent_losses)
        smoothened_action_accuracy = sum(recent_action_accuracies) / len(recent_action_accuracies)
        smoothened_l1_loss = sum(recent_l1_losses) / len(recent_l1_losses)

        # Push Metrics to W&B (every 10 gradient steps)
        if distributed_state.is_main_process and gradient_step_idx % 10 == 0:
            wandb.log(
                {
                    "train_loss": smoothened_loss,
                    "subtrajectory_ID_accuracy": smoothened_action_accuracy,
                    "cross_entropy_loss": smoothened_l1_loss,
                },
                step=gradient_step_idx,
            )

        # Optimizer Step
        if (batch_idx + 1) % grad_accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
            progress.update()

        # Save Model Checkpoint =>> by default, only keeps the latest checkpoint, continually overwriting it!
        if gradient_step_idx > 0 and gradient_step_idx % save_steps == 0:
            if distributed_state.is_main_process:
                print(f"Saving Model Checkpoint for Step {gradient_step_idx}")

                # If LoRA, we first save adapter weights, then merge into full model; otherwise, default save!
                save_dir = adapter_dir if use_lora else run_dir

                # Save Processor & Weights
                processor.save_pretrained(run_dir)
                vla.save_pretrained(save_dir)

            # Wait for processor and adapter weights to be saved by main process
            # dist.barrier()

            # Merge LoRA weights into model backbone for faster inference
            #   =>> Note that merging is slow and can be done post-hoc to speed up training
            if use_lora:
                base_vla = AutoModelForVision2Seq.from_pretrained(
                    vla_path, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, trust_remote_code=True
                )
                merged_vla = PeftModel.from_pretrained(base_vla, adapter_dir)
                merged_vla = merged_vla.merge_and_unload()
                if distributed_state.is_main_process:
                    if save_latest_checkpoint_only:
                        # Overwrite latest checkpoint
                        merged_vla.save_pretrained(run_dir)

                        print(f"Saved Model Checkpoint for Step {gradient_step_idx} at: {run_dir}")
                    else:
                        # Prepare to save checkpoint in new directory
                        checkpoint_dir = Path(str(run_dir) + f"--{gradient_step_idx}_chkpt")
                        os.makedirs(checkpoint_dir, exist_ok=True)

                        # Save dataset statistics to new directory
                        save_dataset_statistics(vla_dataset.dataset_statistics, checkpoint_dir)

                        # Save processor and model weights to new directory
                        processor.save_pretrained(checkpoint_dir)
                        merged_vla.save_pretrained(checkpoint_dir)

                        print(f"Saved Model Checkpoint for Step {gradient_step_idx} at: {checkpoint_dir}")

            # Block on Main Process Checkpointing
            # dist.barrier()

        # Stop training when max_steps is reached
        if gradient_step_idx == max_steps:
            print(f"Max step {max_steps} reached! Stopping training...")
            break

 50%|████████████████████████                        | 501/1000 [03:52<03:42,  2.24it/s]/home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/peft/utils/save_and_load.py:180: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


Saving Model Checkpoint for Step 500


Loading checkpoint shards: 100%|██████████████████████████| 3/3 [00:00<00:00,  3.60it/s]


Saved Model Checkpoint for Step 500 at: runs/openvla-7b+columbia_cairlab_pusht_real+b8+lr-2e-05+test+lora-r32+dropout-0.0


1001it [08:28,  2.24it/s]                                                               /home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/peft/utils/save_and_load.py:180: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


Saving Model Checkpoint for Step 1000


Loading checkpoint shards: 100%|██████████████████████████| 3/3 [00:00<00:00,  3.51it/s]
                         

Saved Model Checkpoint for Step 1000 at: runs/openvla-7b+columbia_cairlab_pusht_real+b8+lr-2e-05+test+lora-r32+dropout-0.0
Max step 1000 reached! Stopping training...


# INFERENCE OPENVLA

In [44]:
import sys
import os

os.chdir("/home/ids/ext-5219/tokenizer/openvla-oft/")
print("Current working directory:", os.getcwd())

sys.argv.append("pusht")

import torch

from PIL import Image
import numpy as np
from pathlib import Path
from prismatic.vla.datasets.datasets_custom import RLDSCustomBatchTransform, RLDSDatasetCustom

from transformers import AutoModelForVision2Seq, AutoProcessor


from experiments.robot.openvla_utils import _load_dataset_stats
from prismatic.vla.subtrajectory_tokenizer import SubtrajectoryTokenizer
from prismatic.models.backbones.llm.prompting import PurePromptBuilder


pretrained_checkpoint ="/home/ids/ext-5219/tokenizer/openvla-oft/runs/openvla-7b+columbia_cairlab_pusht_real+b8+lr-2e-05+test+lora-r32+dropout-0.0"

# pretrained_checkpoint ="/home/ids/ext-5219/tokenizer/openvla-oft/runs/openvla-7b+columbia_cairlab_pusht_real+b8+lr-2e-05+test+lora-r32+dropout-0.0--image_aug"
# Instantiate config


# Load OpenVLA policy and inputs processor
processor = AutoProcessor.from_pretrained(pretrained_checkpoint, trust_remote_code=True)
# vla = AutoModelForVision2Seq.from_pretrained(
#     pretrained_checkpoint, 
#     attn_implementation="flash_attention_2",  # [Optional] Requires `flash_attn`
#     torch_dtype=torch.bfloat16, 
#     low_cpu_mem_usage=True, 
#     trust_remote_code=True
# ).to("cuda:0")


vla = AutoModelForVision2Seq.from_pretrained(
    pretrained_checkpoint,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
).to("cuda:0")

_load_dataset_stats(vla, pretrained_checkpoint)

print("✓ Modèle chargé avec succès !")

# Load dataset via RLDSDataset (same as training pipeline)
data_root_dir: Path = Path("/home/ids/ext-5219/tokenizer/test")
dataset_name: str = "columbia_cairlab_pusht_real"

# Create batch transform
action_tokenizer_inf = SubtrajectoryTokenizer(processor.tokenizer)
batch_transform_inf = RLDSCustomBatchTransform(
    action_tokenizer_inf,
    processor.tokenizer,
    image_transform=processor.image_processor.apply_transform,
    prompt_builder_fn=PurePromptBuilder,
    use_wrist_image=False,
    use_proprio=False,
)

# Create dataset (this properly handles the data structure)
inference_dataset = RLDSDatasetCustom(
    data_root_dir,
    dataset_name,
    batch_transform_inf,
    resize_resolution=tuple(vla.config.image_sizes),
    shuffle_buffer_size=10000,
    image_aug=False,
)

# Get first sample
sample_iterator = iter(inference_dataset)
sample_dict = next(sample_iterator)

print(f"✓ Dataset loaded. Sample keys: {sample_dict.keys()}")

# Extract ground truth subtrajectory_id (check if it exists)

ground_truth_subtrajectory_id = sample_dict["subtraj_ids"]
print(f"✓ Ground truth subtrajectory ID extracted: {ground_truth_subtrajectory_id}")
print(f"✓ Ground truth subtrajectory ID extracted: {ground_truth_subtrajectory_id.shape}")



# Extract and prepare observation for inference
inputs = {
        "pixel_values": sample_dict["pixel_values"].unsqueeze(0).to("cuda:0", dtype=torch.bfloat16),
        "input_ids": sample_dict["input_ids"].unsqueeze(0).to("cuda:0"),
    }

# Generate robot action chunk (sequence of future actions)
print("\n→ Starting inference ...")


action = vla.predict_subtraj_ID(action_tokenizer_inf,processor.tokenizer, **inputs, unnorm_key="columbia_cairlab_pusht_real", do_sample=False)


gt_cluster =ground_truth_subtrajectory_id


# S'assurer que les deux sont des tableaux numpy "plats" pour la comparaison
gt_flat = int(gt_cluster.item())
pred_flat = int(action.item())



# Comparaison avec une tolérance pour les flottants
is_match = gt_flat== pred_flat

print(f"\n{'='*60}")
print(f"INFERENCE RESULTS:")
print(f"{'='*60}")

match = "✓ CORRECT" if is_match else "✗ MISMATCH"

print(f"Ground Truth: {gt_flat}")
print(f"Predicted:    {pred_flat}")
print(f"Result:       {match}")



Current working directory: /home/ids/ext-5219/tokenizer/openvla-oft


Loading checkpoint shards: 100%|█| 4/4 [00:01<00:00,  3.42


✓ Modèle chargé avec succès !


02/03 [18:40:32] INFO     | >> Load dataset info from                                           ]8;id=847909;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/dataset_info.py\dataset_info.py]8;;\:]8;id=802694;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/dataset_info.py#599\599]8;;\
                          /home/ids/ext-5219/tokenizer/test/columbia_cairlab_pusht_real/1.0.0                      

                 INFO     | >> Constructing tf.data.Dataset columbia_cairlab_pusht_real for    ]8;id=515255;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/logging/logging_logger.py\logging_logger.py]8;;\:]8;id=81860;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/logging/logging_logger.py#49\49]8;;\
                          split all, from                                                                          
                          /home/ids/ext-5219/tokenizer/test/columbia_cairlab_pusht_real/1.0.0                      

...

                 INFO     | >> Load dataset info from                                           ]8;id=471666;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/dataset_info.py\dataset_info.py]8;;\:]8;id=216850;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/dataset_info.py#599\599]8;;\
                          /home/ids/ext-5219/tokenizer/test/columbia_cairlab_pusht_real/1.0.0                      

                 INFO     | >> Constructing tf.data.Dataset columbia_cairlab_pusht_real for    ]8;id=966070;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/logging/logging_logger.py\logging_logger.py]8;;\:]8;id=459280;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/logging/logging_logger.py#49\49]8;;\
                          split train, from                                                                        
                          /home/ids/ext-5219/tokenizer/test/columbia_cairlab_pusht_real/1.0.0                      

2026-02-03 18:40:32.616322: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization


                 INFO     | >> [*] Applying frame transforms on dataset...                           ]8;id=655485;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py\dataset.py]8;;\:]8;id=980231;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py#582\582]8;;\

✓ Dataset loaded. Sample keys: dict_keys(['pixel_values', 'input_ids', 'labels', 'dataset_name', 'subtraj_ids'])
✓ Ground truth subtrajectory ID extracted: [8]
✓ Ground truth subtrajectory ID extracted: (1,)

→ Starting inference ...

INFERENCE RESULTS:
Ground Truth: 8
Predicted:    8
Result:       ✓ CORRECT
